# Ripeness Classification with YOLOv8 - Jetson Optimized
This notebook trains YOLOv8 for pineapple ripeness classification and saves weights for Jetson deployment.

In [ ]:
import sys
sys.path.insert(0, '../config')
sys.path.insert(0, '..')

from paths import *
from jetson_utils import quantize_yolo_model, convert_to_onnx, get_model_info
from ultralytics import YOLO
import yaml

In [ ]:
# Create data configuration for YOLO classification
print(f'Classification data path: {CLASSIFICATION_DATA}')

data_config = {
    'path': str(CLASSIFICATION_DATA / 'molvumClassification'),
    'train': str(CLASSIFICATION_TRAIN),
    'test': str(CLASSIFICATION_TEST),
}

yaml_path = '../config/classification_data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)
print(f'✓ Data configuration saved to {yaml_path}')

In [ ]:
# Install/update required packages
import subprocess
import sys

packages = ['ultralytics>=8.0.0', 'opencv-python', 'torch', 'torchvision']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('✓ Dependencies installed')

In [ ]:
# Load YOLOv8 classification model
print('Loading YOLOv8 Classification model...')
model = YOLO('yolov8n-cls.pt')  # nano classification model
print(f'Model loaded successfully')

In [ ]:
# Train the classification model
print('Starting training...')
results = model.train(
    data=str(CLASSIFICATION_DATA / 'molvumClassification'),
    epochs=100,
    imgsz=224,
    batch=32,
    lr0=0.001,
    weight_decay=0.0005,
    patience=20,
    save=True,
    device=0
)
print('✓ Training completed')

In [ ]:
import shutil
from pathlib import Path

# Copy best model to models directory
best_model_paths = list(Path('runs/classify').glob('**/best.pt'))
if best_model_paths:
    best_model_path = best_model_paths[-1]
    shutil.copy(str(best_model_path), str(CLASSIFICATION_MODEL))
    print(f'✓ Model saved to {CLASSIFICATION_MODEL}')
    print(f'  Model size: {CLASSIFICATION_MODEL.stat().st_size / 1024 / 1024:.2f} MB')
else:
    print('⚠ Could not find best.pt model')

In [ ]:
# Load and evaluate the best model
model = YOLO(str(CLASSIFICATION_MODEL))
print('Running validation...')
metrics = model.val()
print(f'✓ Validation completed')

In [ ]:
import time

print('Testing inference speed...')
test_images = list(CLASSIFICATION_TEST.glob('**/*.jpg'))[:20]

model = YOLO(str(CLASSIFICATION_MODEL))
start_time = time.time()
for img_path in test_images:
    model.predict(source=str(img_path), verbose=False)
end_time = time.time()

avg_time = (end_time - start_time) / len(test_images)
print(f'\n✓ Performance:')
print(f'  Average inference time: {avg_time:.4f}s ({avg_time*1000:.2f}ms per image)')
print(f'  FPS: {1/avg_time:.2f}')

In [ ]:
print('Preparing model for Jetson deployment...')
print(f'Original model: {CLASSIFICATION_MODEL.stat().st_size / 1024 / 1024:.2f} MB')

# Export to FP16 for Jetson
model = YOLO(str(CLASSIFICATION_MODEL))
fp16_export = model.export(format='torchscript', half=True, device=0)
print(f'✓ FP16 model exported')

# Export to ONNX
onnx_export = model.export(format='onnx', opset=13, device=0)
print(f'✓ ONNX model exported')

get_model_info(CLASSIFICATION_MODEL)

## Summary
✓ Classification model trained and saved
✓ Quantized for Jetson deployment
✓ Ready for real-time inference